# SMCtrl_Phase0_SHAP and Feature Engineering

### You should run 0.Preprocessing in Full_dataAnalysis.ipynb and generate "1_chopped" folder before running the code in this notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import statsmodels.api as sm
import struct
import random
import re

from smctrl import phase, HELPER, kinematics, condition_mapping
from smctrl.scripts import run_chop_valid_data, run_collision_correction, run_timerectify_resample_and_filter

%load_ext autoreload
%autoreload 2

In [ ]:
current_phase = phase.Phase("phase0")

# overall condition dictionary, dont change

condition_dict={
    0: (0, 1.7, 1.5, 1),
    1: (0, 1.7 ,2, 1),
    2: (0, 1.7 ,2.5, 1),
    3: (0, 1.7 ,2, 0.38),
    4: (0, 1.7 ,2.5, 0.38),
    5: (0, 1.7 ,2, 1.4),
    6: (0, 1.7 ,2.5, 1.4)
}

# Subject to analyze
sub_nums = [11]

base = current_phase.data_dir / "1_chopped"

In [ ]:
for sub in sub_nums:
    print(f"sub {sub}:")
    for sub_dir in sorted(base.glob(f"sub{sub}_*")):
        for f in sorted(sub_dir.glob("marker_order*_cond*.csv")):
            # ============================================================
            # 1. lock cond
            # ============================================================
            match = re.search(r'_cond(\d+)', f.name)
            if match:
                cond = int(match.group(1))
                out_folder = sub_dir / f"cond{cond}"
                out_folder.mkdir(parents=True, exist_ok=True)
                print(f"cond {cond}")
            else:
                raise ValueError("No conds found")
            
            # ============================================================
            # 2. read csv as df
            # ============================================================
            feature_dict = {
                "HMD": [
                    "HandMaxVelocityX",
                    "HandMaxVelocityY",
                    "HandMaxVelocityZ"
                ],
                "FINGER": [
                    "finger_0_spd",
                    "finger_1_spd",
                    "finger_2_spd",
                    "finger_3_spd",
                    "finger_4_spd"
                ],
                "PUPIL": [
                    "Left Pupil Diameter (mm)",
                    "Right Pupil Diameter (mm)"
                ],
                "GAZE": [
                    "TargetRelX",
                    "TargetRelY"
                ],
                "EMG": [
                    "Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8"
                ],
                "EEG": [
                    "Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7"
                ]
            }

            time_col_dict={
                "HMD": ["UnixTime(ms)", "LocalTime"],
                "FINGER":"unix_time",
                "PUPIL": "Capture Unix timestamp",
                "GAZE": "UnixTime(ms)",
                "EMG": "Timestamp(ms)",
                "EEG": "Timestamp(ms)"
            }

            modality_freq_dict={
                "HMD":90,
                "FINGER":120,
                "PUPIL": 200,
                "GAZE": 200,
                "EMG": 250,
                "EEG": 250
            }

            merged_dict = {}
            for key in feature_dict.keys():
                tcols = time_col_dict[key]
                merged_dict[key] = (tcols if isinstance(tcols, list) else [tcols]) + feature_dict[key]

            dart_file_name   = f.with_name(f.name.replace("marker", "dart"))
            hmd_file_name    = f.with_name(f.name.replace("marker", "hmd"))
            finger_file_name = f.with_name(f.name.replace("marker", "hand_position"))
            pupil_file_name  = f.with_name(f.name.replace("marker", "eye"))
            gaze_file_name   = f.with_name(f.name.replace("marker", "gaze"))
            eeg_file_name    = f.with_name(f.name.replace("marker", "eeg"))
            emg_file_name    = f.with_name(f.name.replace("marker", "emg"))
            result_file_name = f.with_name(f.name.replace("marker", "result"))

            try:
                df_hmd = pd.read_csv(hmd_file_name, usecols=merged_dict["HMD"])
                df_finger = pd.read_csv(finger_file_name, usecols=merged_dict["FINGER"])
                df_pupil = pd.read_csv(pupil_file_name, usecols=merged_dict["PUPIL"])
                df_gaze = pd.read_csv(gaze_file_name, usecols=merged_dict["GAZE"])
                df_eeg = pd.read_csv(eeg_file_name, usecols=merged_dict["EEG"])
                df_emg = pd.read_csv(emg_file_name, usecols=merged_dict["EMG"])
                df_results = pd.read_csv(result_file_name)
                df_markers = pd.read_csv(f)
            except Exception as e:
                print(f"Error while reading CSV file: {e}")
                raise

            # ============================================================
            # 3. Pre-processing Pipeline
            #
            # EMG:
            #   - Band-pass filter: [20, 120] Hz
            #   - Notch filter: [60] Hz
            #   - Amplitude threshold: ±3000 µV
            #
            # EEG:
            #   - Band-pass filter: [1, 120] Hz
            #   - Notch filter: [60] Hz
            #   - Amplitude threshold: ±200 µV
            #
            # PUPIL:
            #   - Band-pass filter: [0.01, 4.0] Hz
            #
            # FINGER:
            #   - Low-cut filter: 5.0 Hz
            # ============================================================
            # ============================================================
            # 3.1 Timestamp Adjustment
            #
            # - Convert all time columns to milliseconds (ms)
            # - Align multi-modal timestamps (hand, finger, EMG, EEG, pupil)
            # - Estimate average sampling rate for each modality
            # - Resample to uniform time grid using interpolation
            #     * Hand:   90 Hz  → 11.11 ms
            #     * Finger: 120 Hz → 8.33 ms
            #     * Eye:    200 Hz → 5.00 ms
            #     * EMG:    250 Hz → 4.00 ms
            #     * EEG:    250 Hz → 4.00 ms
            # ============================================================
            df_finger["UnixTime(ms)"] = df_finger["unix_time"] / 1e6
            df_pupil["UnixTime(ms)"] = df_pupil["Capture Unix timestamp"] / 1e6

            # hmd
            print("processing hmd...")
            df_hmd = df_hmd.sort_values(time_col_dict["HMD"]).reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_hmd, time_col=time_col_dict["HMD"][0])
            resampled_segments = [HELPER.resample_df(seg, time_col=time_col_dict["HMD"][0], fs_target=modality_freq_dict["HMD"]) for seg in segments]
            filtered_segments = resampled_segments
            df_hmd_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_hmd_norm = HELPER.normalize(df_hmd_filtered, feat_cols=feature_dict["HMD"], method="zscore")

            # pupil
            print("processing eye...")
            df_pupil = df_pupil.sort_values("UnixTime(ms)").reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_pupil, time_col="UnixTime(ms)")
            resampled_segments = [HELPER.resample_df(seg, time_col="UnixTime(ms)", fs_target=modality_freq_dict["PUPIL"]) for seg in segments]
            filtered_segments = [
                seg.assign(**{
                    col: HELPER.bandpass_filter(seg[col].to_numpy(), lowcut=0.01, highcut=5.0, fs=modality_freq_dict["PUPIL"])
                    for col in feature_dict["PUPIL"]
                })
                for seg in resampled_segments
            ]
            df_pupil_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_pupil_norm = HELPER.normalize(df_pupil_filtered, feat_cols=feature_dict["PUPIL"], method="robust_zscore")

            # gaze
            print("processing gaze...")
            df_gaze = df_gaze.sort_values(time_col_dict["GAZE"]).reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_gaze, time_col=time_col_dict["GAZE"])
            resampled_segments = [HELPER.resample_df(seg, time_col=time_col_dict["GAZE"], fs_target=modality_freq_dict["GAZE"]) for seg in segments]
            filtered_segments = resampled_segments
            df_gaze_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_gaze_norm = HELPER.normalize(df_gaze_filtered, feat_cols=feature_dict["GAZE"], method="zscore")

            # finger
            print("processing finger...")
            df_finger = df_finger.sort_values("UnixTime(ms)").reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_finger, time_col="UnixTime(ms)")
            resampled_segments = [HELPER.resample_df(seg, time_col="UnixTime(ms)", fs_target=modality_freq_dict["FINGER"]) for seg in segments]
            filtered_segments = [
                seg.assign(**{
                    col: HELPER.bandpass_filter(seg[col].to_numpy(), highcut=5.0, fs=modality_freq_dict["FINGER"])
                    for col in feature_dict["FINGER"]
                })
                for seg in resampled_segments
            ]
            df_finger_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_finger_norm = HELPER.normalize(df_finger_filtered, feat_cols=feature_dict["FINGER"], method="zscore")

            # eeg
            print("processing eeg...")
            df_eeg = df_eeg.sort_values(time_col_dict["EEG"]).reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_eeg, time_col=time_col_dict["EEG"])
            resampled_segments = [HELPER.resample_df(seg, time_col=time_col_dict["EEG"], fs_target=modality_freq_dict["EEG"]) for seg in segments]
            filtered_segments = [
                seg.assign(**{
                    col: HELPER.bandpass_filter(seg[col].to_numpy(), lowcut=1.0, highcut=80.0, fs=modality_freq_dict["EEG"])
                    for col in feature_dict["EEG"]
                })
                for seg in resampled_segments
            ]
            df_eeg_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_eeg_norm = HELPER.normalize(df_eeg_filtered, feat_cols=feature_dict["EEG"], method="max")

            # emg
            print("processing emg...")
            df_emg = df_emg.sort_values(time_col_dict["EMG"]).reset_index(drop=True)
            segments = HELPER.segment_by_gap(df_emg, time_col=time_col_dict["EMG"])
            resampled_segments = [HELPER.resample_df(seg, time_col=time_col_dict["EMG"], fs_target=modality_freq_dict["EMG"]) for seg in segments]
            filtered_segments = [
                seg.assign(**{
                    col: HELPER.bandpass_filter(seg[col].to_numpy(), lowcut=20.0, fs=modality_freq_dict["EMG"])
                    for col in feature_dict["EMG"]
                })
                for seg in resampled_segments
            ]
            df_emg_filtered = pd.concat(filtered_segments, ignore_index=True)
            df_emg_norm = HELPER.normalize(df_emg_filtered, feat_cols=feature_dict["EMG"], method="max")

            # markers
            print("processing results...")
            fit = np.polyfit(df_hmd_norm["UnixTimeRaw(ms)"], df_hmd_norm["UnixTimeCorr(ms)"], 1)
            a, b = fit
            df_markers["UnixTimeCorr(ms)"] = a * df_markers["UnixTime(ms)"] + b

            # results
            df_results = HELPER.calculate_corrected_hit_point(df_results)

            # ============================================================
            # 4. chop data corrding to marker and bin
            # ============================================================
            BEFORE_PINCH_BINS = 3
            AFTER_PINCH_BINS = 4

            # find pass-release
            init_hit_times = df_markers["TotalHitTimes"].iloc[0]
            hit_markers = df_markers.loc[df_markers["Marker"] == "Hit"].copy()
            hit_idx = hit_markers.index
            pass_markers = df_markers.iloc[hit_idx - 3]
            pinch_markers = df_markers.iloc[hit_idx - 2]
            release_markers = df_markers.iloc[hit_idx - 1]
            available_trials = (pinch_markers["UnixTimeCorr(ms)"].values - pass_markers["UnixTimeCorr(ms)"].values > (1000*2*BEFORE_PINCH_BINS/modality_freq_dict["HMD"])) & (release_markers["UnixTimeCorr(ms)"].values - pinch_markers["UnixTimeCorr(ms)"].values > (1000*2*AFTER_PINCH_BINS/modality_freq_dict["HMD"]))
            print(f"available trials for binning: {available_trials.sum()} / {release_markers.shape[0]}")

            def bin_df(df, modality_name, pass_df, pinch_df, release_df):
                assert modality_name in ["HMD", "PUPIL", "FINGER", "GAZE", "EEG", "EMG"], "unknown feature"
                pass_pinch = df.loc[(df["UnixTimeCorr(ms)"] >= pass_df["UnixTimeCorr(ms)"]) 
                                    & (df["UnixTimeCorr(ms)"] <= pinch_df["UnixTimeCorr(ms)"])].copy()
                pinch_release = df.loc[(df["UnixTimeCorr(ms)"] >= pinch_df["UnixTimeCorr(ms)"]) 
                                    & (df["UnixTimeCorr(ms)"] <= release_df["UnixTimeCorr(ms)"])].copy()
                
                if pass_pinch.shape[0] <= 2*BEFORE_PINCH_BINS or pinch_release.shape[0] <= 2*AFTER_PINCH_BINS:
                    return None

                binned_pass_pinch = HELPER.bin_features(
                    pass_pinch,
                    time_col="UnixTimeCorr(ms)",
                    feat_cols=feature_dict[modality_name],
                    n_bins=BEFORE_PINCH_BINS,
                    agg="max"
                )

                binned_pinch_release = HELPER.bin_features(
                    pinch_release,
                    time_col="UnixTimeCorr(ms)",
                    feat_cols=feature_dict[modality_name],
                    n_bins=AFTER_PINCH_BINS,
                    agg="max"
                )
                assert binned_pass_pinch.shape[0] == BEFORE_PINCH_BINS, f"{modality_name} before frame has nan value"
                assert binned_pinch_release.shape[0] == AFTER_PINCH_BINS, f"{modality_name} after frame has nan value"

                return pd.concat([binned_pass_pinch, binned_pinch_release], ignore_index=True)

            for (_, pass_row), (_, pinch_row), (_, release_row) in zip(pass_markers.iterrows(), pinch_markers.iterrows(), release_markers.iterrows()):
                assert pass_row["TotalHitTimes"] == release_row["TotalHitTimes"], "release and pinch hit count doesn't match"

                dart_order = pinch_row["TotalHitTimes"] - init_hit_times

                result = df_results.loc[df_results["TotalHitTimes"] == pinch_row["TotalHitTimes"], "HitObjectTag"].iloc[0]
                bi_result = 1 if result == "Target" else 0

                continuous_result = df_results.loc[df_results["TotalHitTimes"] == pinch_row["TotalHitTimes"], "Radial_Error"].iloc[0]
                
                all_modalities = ["HMD", "PUPIL", "FINGER", "GAZE", "EEG", "EMG"]
                binned_results = {}

                for modality in all_modalities:
                    df_mod = eval(f"df_{modality.lower()}_norm")
                    binned = bin_df(df_mod, modality, pass_row, pinch_row, release_row)
                    if binned is None:
                        print(f"Skip trial {dart_order}: {modality} segment too short")
                        break
                    binned_results[modality] = binned.reset_index(drop=True)

                else:
                    binned_df = pd.concat(
                        [df.add_prefix(f"{modality}_") for modality, df in binned_results.items()],
                        axis=1
                    ).reset_index(drop=True)
                    binned_df["TrialID"] = dart_order
                    binned_df["bin_label"] = bi_result
                    binned_df["continuous_label"] = continuous_result
                    out_file = out_folder / f"{dart_order}.csv"
                    binned_df.to_csv(out_file, index=False)

sub 11:
cond 0
processing hmd...
Original fs ≈ 83.33 Hz → Resampled to 90.00 Hz
processing eye...
Original fs ≈ 199.99 Hz → Resampled to 200.00 Hz
processing gaze...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing finger...
Original fs ≈ 120.11 Hz → Resampled to 120.00 Hz
Original fs ≈ 118.66 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.25 Hz → Resampled to 120.00 Hz
Original fs ≈ 118.88 Hz → Resampled to 120.00 Hz
Original fs ≈ 124.07 Hz → Resampled to 120.00 Hz
Original fs ≈ 117.87 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.41 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.76 Hz → Resampled to 120.00 Hz
processing eeg...
Original fs ≈ 250.00 Hz → Resampled to 250.00 Hz
Original fs ≈ 250.00 Hz → Resampled to 250.00 Hz
processing emg...
Original fs ≈ 250.06 Hz → Resampled to 250.00 Hz
Original fs ≈ 125.03 Hz → Resampled to 250.00 Hz
Original fs ≈ 250.06 Hz → Resampled to 250.00 Hz
processing results...
available trials for binning: 15 / 15
cond 1
processing hmd...
Origin

e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encount

processing eeg...
Original fs ≈ 250.02 Hz → Resampled to 250.00 Hz
processing emg...
Original fs ≈ 250.02 Hz → Resampled to 250.00 Hz
processing results...
available trials for binning: 26 / 26
cond 2
processing hmd...
Original fs ≈ 83.33 Hz → Resampled to 90.00 Hz
processing eye...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing gaze...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing finger...
Original fs ≈ 119.74 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.44 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.69 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.85 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.61 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.99 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.72 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.47 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.43 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.14 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.51 Hz → Resampled to 120.00 Hz
Original fs ≈ 117.99 Hz → Resampled to 

e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encount

cond 3
processing hmd...
Original fs ≈ 83.33 Hz → Resampled to 90.00 Hz
processing eye...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing gaze...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing finger...
Original fs ≈ 119.29 Hz → Resampled to 120.00 Hz
Original fs ≈ 122.45 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.00 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.03 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.67 Hz → Resampled to 120.00 Hz
Original fs ≈ 118.06 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.00 Hz → Resampled to 120.00 Hz
Original fs ≈ 120.94 Hz → Resampled to 120.00 Hz
processing eeg...
Original fs ≈ 250.00 Hz → Resampled to 250.00 Hz
processing emg...
Original fs ≈ 250.02 Hz → Resampled to 250.00 Hz
processing results...
available trials for binning: 19 / 20
Skip trial 1: HMD segment too short
cond 4
processing hmd...
Original fs ≈ 83.33 Hz → Resampled to 90.00 Hz
processing eye...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing 

e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
e:\anaconda3\envs\SMCtrl\Lib\site-packages\numpy\_core\_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
e:\anaconda3\envs\SMCtrl\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encount

processing eeg...
Original fs ≈ 250.00 Hz → Resampled to 250.00 Hz
processing emg...
Original fs ≈ 250.02 Hz → Resampled to 250.00 Hz
processing results...
available trials for binning: 27 / 28
Skip trial 13: HMD segment too short
Skip trial 18: FINGER segment too short
cond 6
processing hmd...
Original fs ≈ 83.33 Hz → Resampled to 90.00 Hz
processing eye...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing gaze...
Original fs ≈ 200.00 Hz → Resampled to 200.00 Hz
processing finger...
Original fs ≈ 122.10 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.34 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.75 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.35 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.14 Hz → Resampled to 120.00 Hz
Original fs ≈ 119.85 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.70 Hz → Resampled to 120.00 Hz
Original fs ≈ 121.53 Hz → Resampled to 120.00 Hz
Original fs ≈ 117.77 Hz → Resampled to 120.00 Hz
Original fs ≈ 124.11 Hz → Resampled to 120.00 Hz
Original fs